&nbsp;

# Protein-coding Transcript - lncRNA transcript Common CRE

- protein-coding transcript と lncRNA transcript で共通する CREを持つエントリーを探す
- up5p transcripts と、down5p transcripts で同じ操作を行うので、この1つの notebook で処理する
- [../01_HNscore/01_3_HNscore_transcript_gencode_annotation.ipynb](../01_HNscore/01_3_HNscore_transcript_gencode_annotation.ipynb) で、GTFファイルの `strand` や `tags`を統合しているので、こちらでの操作は必要ない

In [1]:
import os
import datetime
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import polars as pl
import plotly.express as px
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from IPython.display import display

pl.Config.set_fmt_str_lengths(300)
os.environ["POLARS_MAX_THREADS"] = "16"

&nbsp;

&nbsp;

&nbsp;

# 1. Upregulated Transcripts (up5p)

&nbsp;

## 1-1: up5p fantabio

- これまでは、[../04_ChIPatlas_diff/up5p_diff_analysis_atac.ipynb](../04_ChIPatlas_diff/up5p_diff_analysis_atac.ipynb) だったが、CRE との統合したファイルを input にする

In [2]:
up5p_fantabio_cre = pl.read_parquet(
    "../../out/fantabio/parquet_file/up5p_transcript_fantabio_v261001.parquet"
)

pl.Config.set_tbl_rows(2)
print(f"protein-coding transcript - CRE pair count: {up5p_fantabio_cre.filter(pl.col('transcript_type') == 'protein_coding').select(['cre_id', 'transcript_id']).n_unique()}")
print(f"lncRNA transcript - CRE pair count: {up5p_fantabio_cre.filter(pl.col('transcript_type') == 'lncRNA').select(['cre_id', 'transcript_id']).n_unique()}")
display(up5p_fantabio_cre)

protein-coding transcript - CRE pair count: 35755
lncRNA transcript - CRE pair count: 9221


transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,cre_chrom,cre_chrom_start,cre_chrom_end,cre_name,tss_nearest_transcript_ids,tss_nearest_transcript_distance,transcript_distance,screen_ccres,chip_atlas_antigens,silencerdbs,fantom5_cage_peaks,fantom5_enhancers,directionality,fantabio_cre_class,itemRgb,itemRgb_name
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,str,i64,str,str,list[str],str,i64,list[struct[2]],list[struct[3]],list[struct[5]],list[struct[2]],list[struct[1]],f64,str,str,str
"""ENST00000309758.6""",352,17,37,335,232,4,15,228,"""HSPA6""","""HGNC:5239""","""protein_coding""","""ENSG00000173110.8""","[""basic"", ""Ensembl_canonical"", … ""CCDS""]","""+""",6,null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""MANE_select""]",1,"""FCHS_28105""","""chr1""",161523745,"""161524487""","""cp1@HSPA6""","[""GQ891451""]","""0""",53,"[{""EH38E2843853"",""pELS""}, {""EH38E2843855"",""PLS""}, {""EH38E3981533"",""pELS""}]","[{""STAG2"",4103,[{""SRX9975614"",4103}, {""SRX9975613"",3500}, {""SRX26159208"",562}]}, {""HSF1"",4038,[{""SRX16646095"",4038}, {""SRX16646111"",3572}, … {""SRX13845684"",554}]}, … {""GABPA"",500,[{""SRX2635792"",500}]}]","[{""HS_00013400"",""H1 BMP4 Derived Mesendoderm Cultured Cells"",""Mesendoderm cell"",""Others"",""SVM""}, {""HS_00013401"",""H1 Cells"",""Embryonic stem cell"",""Stem cell"",""SVM""}, … {""HS_00013403"",""Foreskin Fibroblast Primary Cells skin02"",""Prepuce"",""Skin"",""SVM""}]",null,null,0.982111,"""PLA""","""204,0,0""","""red"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000774581.1""",101,57,248,44,34,44,173,-10,null,null,"""lncRNA""","""ENSG00000300853.1""","[""basic"", ""TAGENE""]","""+""",2,"""URS0002A25289""","""ENSEMBL_GENCODE""",null,"""havana_tagene_homo_sapiens""","[""CLS3 project""]",1,"""FCHS_400104""","""chr8""",27592922,"""27593122""","""FCHS_400104""","[""ENST00000774579.1""]","""0""",294,"[{""EH38E3824983"",""dELS""}]",null,"[{""HS_00105821"",""NHEK-Epidermal Keratinocyte Primary Cells"",""Skin"",""Skin"",""SVM""}, {""HS_00105822"",""H1 BMP4 Derived Trophoblast Cultured Cells"",""Trophoblast cell"",""Others"",""SVM""}, … {""HS_00105828"",""Breast variant Human Mammary Epithelial Cells (vHMEC)"",""Breast"",""Breast"",""SVM""}]",null,null,0.5,"""ELA""","""255,202,74""","""yellow"""


&nbsp;

## 1-2: Select Columns

In [3]:
column_list = [
    "cre_id",
    "cre_name",
    "fantabio_cre_class",
    "cre_chrom",
    "cre_chrom_start",
    "cre_chrom_end",
    "directionality",
    "transcript_distance",
    "screen_ccres",
    "fantom5_enhancers",
    "transcript_id",
    "HN2",
    "gene_id",
    "gene_symbol",
    "transcript_type",
    "RNACentral_ID",
    "strand",
    "tags",
]

&nbsp;

## 1-3: Protein-coding Transcript

- CRE 側に付与される情報として加工する

In [4]:
pc_transcript = up5p_fantabio_cre.filter(
    pl.col("transcript_type") == "protein_coding"
).select(
    column_list
).unique(
    subset=["cre_id", "transcript_id"]
).with_columns(
    # Pack transcript-side fields into a struct; row grain (CRE x transcript) stays the same
    pl.struct(
        "gene_id",
        "gene_symbol",
        "transcript_type",
        "RNACentral_ID",
        "tags",
    ).alias("pc_transcript_info")
).drop(
    "gene_id",
    "gene_symbol",
    "transcript_type",
    "RNACentral_ID",
    "tags"
).rename(
    {
        "transcript_id": "pc_transcript_id",
        "strand": "pc_transcript_strand",
        "transcript_distance": "pc_transcript_distance",
        "HN2": "pc_HN2"
    }
).sort(
    "pc_HN2",
    descending=True
)


print(f"unique protein-coding transcript count: {pc_transcript.select(pl.col('pc_transcript_id')).n_unique()}")
display(pc_transcript)

unique protein-coding transcript count: 11434


cre_id,cre_name,fantabio_cre_class,cre_chrom,cre_chrom_start,cre_chrom_end,directionality,pc_transcript_distance,screen_ccres,fantom5_enhancers,pc_transcript_id,pc_HN2,pc_transcript_strand,pc_transcript_info
str,str,str,str,i64,str,f64,i64,list[struct[2]],list[struct[1]],str,i64,str,struct[5]
"""FCHS_28105""","""cp1@HSPA6""","""PLA""","""chr1""",161523745,"""161524487""",0.982111,53,"[{""EH38E2843853"",""pELS""}, {""EH38E2843855"",""PLS""}, {""EH38E3981533"",""pELS""}]",null,"""ENST00000309758.6""",335,"""+""","{""ENSG00000173110.8"",""HSPA6"",""protein_coding"",null,[""basic"", ""Ensembl_canonical"", … ""CCDS""]}"
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_392915""","""cp1@ADCK2,cp3@DENND2A""","""ELA""","""chr7""",140673711,"""140674383""",-0.410256,-393,"[{""EH38E2596561"",""pELS""}, {""EH38E3804528"",""pELS""}, {""EH38E3804529"",""pELS""}]",null,"""ENST00000896699.1""",44,"""+""","{""ENSG00000133597.12"",""ADCK2"",""protein_coding"",null,[""basic"", ""TAGENE""]}"


&nbsp;

## 1-4: lncRNA transcript

In [5]:
lncrna_transcript = up5p_fantabio_cre.filter(
    pl.col("transcript_type") == "lncRNA"
).select(
    column_list
).unique(
    subset=["cre_id", "transcript_id"]
).with_columns(
    # Pack transcript-side fields into a struct; row grain (CRE x transcript) stays the same
    pl.struct(
        "gene_id",
        "gene_symbol",
        "transcript_type",
        "RNACentral_ID",
        "tags",
    ).alias("lncrna_transcript_info")
).drop(
    "gene_id",
    "gene_symbol",
    "transcript_type",
    "RNACentral_ID",
    "tags"
).rename(
    {
        "transcript_id": "lncrna_transcript_id",
        "strand": "lncrna_transcript_strand",
        "transcript_distance": "lncrna_transcript_distance",
        "HN2": "lncrna_HN2"
    }
).sort(
    "lncrna_HN2",
    descending=True
)

print(f"unique lncRNA transcript count: {lncrna_transcript.select(pl.col('lncrna_transcript_id')).n_unique()}")
display(lncrna_transcript)

unique lncRNA transcript count: 3657


cre_id,cre_name,fantabio_cre_class,cre_chrom,cre_chrom_start,cre_chrom_end,directionality,lncrna_transcript_distance,screen_ccres,fantom5_enhancers,lncrna_transcript_id,lncrna_HN2,lncrna_transcript_strand,lncrna_transcript_info
str,str,str,str,i64,str,f64,i64,list[struct[2]],list[struct[1]],str,i64,str,struct[5]
"""FCHS_28129""","""cp1@HSPA7""","""ELA""","""chr1""",161605881,"""161606178""",1.0,0,"[{""EH38E2843898"",""pELS""}]",null,"""ENST00000850140.1""",311,"""+""","{""ENSG00000310471.1"",null,""lncRNA"",""URS0002A15385"",[""basic"", ""Ensembl_canonical"", … ""TAGENE""]}"
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_424744""","""FCHS_424744""","""PLA""","""chr9""",98045272,"""98045782""",0.990756,0,"[{""EH38E3896748"",""dELS""}, {""EH38E4501977"",""dELS""}]",null,"""ENST00000752754.1""",44,"""+""","{""ENSG00000298058.1"",null,""lncRNA"",""URS0002A4D370"",[""basic"", ""Ensembl_canonical"", … ""TAGENE""]}"


&nbsp;

&nbsp;

## 1-5: Common CRE

- 作成したデータフレームを `join` することで共通する CRE を特定する
- ここで調べることができるのは、protein-coding transcript と lncRNA transcript 両方が `upregulated` として分類されているペア
- 簡易的な指標として、protein-coding transcript と lncRNA transcript の HN-score の値の和を計算しておく

In [6]:
common_cre = pc_transcript.join(
    lncrna_transcript,
    on=[
        "cre_id",
        "cre_name",
        "fantabio_cre_class",
        "cre_chrom",
        "cre_chrom_start",
        "cre_chrom_end",
        "screen_ccres",
        "fantom5_enhancers",
        "directionality"
    ],
    how="inner",
    coalesce=True,
    nulls_equal=True # これを指定しないと、strand が `null` のエントリーが `join` されない
).with_columns(
    pl.when(
        pl.col("pc_transcript_strand") == pl.col("lncrna_transcript_strand")
    ).then(
        pl.lit("same strand")
    ).otherwise(
        pl.when(
            # explicitly check for strand mismatch
            pl.col("pc_transcript_strand") != pl.col("lncrna_transcript_strand")
        ).then(
            pl.lit("opposite strand")
        ).otherwise(
            pl.lit("unknown strand")
        )
    ).alias("strand_type")
).with_columns(
    (pl.col("pc_HN2") + pl.col("lncrna_HN2")).alias("pc_lncrna_HN2") # 簡易的な指標として、HN2 の和を計算
).cast(
    # なぜか Int64 になっていないので、Int64 に変換
    {
        "cre_chrom_end": pl.Int64
    }
)

print(f"unique CRE count (including PLA and ELA): {common_cre.select(['cre_id']).n_unique()}")
display(common_cre)

unique CRE count (including PLA and ELA): 1087


cre_id,cre_name,fantabio_cre_class,cre_chrom,cre_chrom_start,cre_chrom_end,directionality,pc_transcript_distance,screen_ccres,fantom5_enhancers,pc_transcript_id,pc_HN2,pc_transcript_strand,pc_transcript_info,lncrna_transcript_distance,lncrna_transcript_id,lncrna_HN2,lncrna_transcript_strand,lncrna_transcript_info,strand_type,pc_lncrna_HN2
str,str,str,str,i64,i64,f64,i64,list[struct[2]],list[struct[1]],str,i64,str,struct[5],i64,str,i64,str,struct[5],str,i64
"""FCHS_373931""","""cp4@ZFAND2A,cp1@ZFAND2A-DT""","""PLA""","""chr7""",1160610,1160913,-0.984546,432,"[{""EH38E3752326"",""PLS""}]",null,"""ENST00000401903.6""",317,"""-""","{""ENSG00000178381.14"",""ZFAND2A"",""protein_coding"",null,[""not_organism_supported"", ""basic"", … ""CCDS""]}",-178,"""ENST00000662926.1""",130,"""+""","{""ENSG00000229043.4"",""ZFAND2A-DT"",""lncRNA"",""URS0000EF6DEC"",[""TAGENE""]}","""opposite strand""",447
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_212037""","""cp2@KLK6""","""PLA""","""chr19""",50968585,50969315,-0.997276,389,"[{""EH38E3314744"",""PLS""}, {""EH38E3314745"",""pELS""}]",null,"""ENST00000594641.1""",44,"""-""","{""ENSG00000167755.16"",""KLK6"",""protein_coding"",null,[""alternative_5_UTR"", ""upstream_ATG"", … ""CCDS""]}",-392,"""ENST00000671133.1""",50,"""+""","{""ENSG00000267879.2"",null,""lncRNA"",""URS0000EF307B"",[""basic"", ""TAGENE""]}","""opposite strand""",94


&nbsp;

## 1-6: Integration Resources

- 下流の解析で flag などに使用するデータをここで読み込んでおく

&nbsp;

1. ensembl promoter window (GENCODE release 48) - fanta.bio mapping file
2. GENCODE release 49 vs FANTOM-CAT (stringent) ID mapping file

&nbsp;

## 1-6-1: ensembl promoter window (GENCODE release 48) - fanta.bio mapping file

In [8]:
ensembl_promoter_intersect = pl.read_csv(
    "../../out/fantabio/ensembl_promoter/gencode.v48.promoter_windows_v260908_intersecting_human-CREv1.2.1.hg38.cre-peaks_v261003_one_per_promoter.tsv",
    separator="\t"
)

display(ensembl_promoter_intersect)

chromosome_ensembl,start_ensembl,end_ensembl,ensembl_promoter_accession,strand,chromosome_fantabio,start_fantabio,end_fantabio,cre_id,cre_name,intersect_length,fantabio_cre_overlap_frac_of_ensembl_promoter_window,directionality,fantabio_cre_class
str,i64,i64,str,str,str,i64,i64,str,str,i64,f64,f64,str
"""chrX""",100636806,100637806,"""ENSG00000000003_promoter_window""","""-""","""chrX""",100636532,100637120,"""FCHS_442038""","""cp4@TSPAN6""",314,0.314,-0.998632,"""PLA"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chr19""",1274529,1275529,"""ENSG00000310576_promoter_window""","""+""","""chr19""",1275363,1275715,"""FCHS_196119""","""cp3@FAM174C""",166,0.166,0.991043,"""PLA"""


&nbsp;

## 1-6-2. GENCODE release 49 vs FANTOM-CAT (stringent) ID mapping file

In [9]:
gencode_v49_v_fantomcat_stringent_id_mapping = pl.read_csv(
    "../../out/GENCODE_FANTOMCAT_comparison/GENCODE_v49_FANTOMCAT_lv4_stringent_comparison_v261010.tsv",
    separator="\t"
).rename(
    {
        "GENCODE_transcript": "lncrna_transcript_id"
    }
).group_by(
    "lncrna_transcript_id",
    "id_correspondence_flag",
    "Transcript_classification_codes",
    "FANTOM-CAT_transcript_lv4_stringent",
    "gene class"
).agg(
    pl.struct(
        "FANTOM-CAT_gene_lv4_stringent",
        "DHS support",
        "TIR conservation[RS score]",
        "exon conservation[RS score]",
        "trait vs cell enrichment[fdr]",
        "eQTL mRNA coexpression[fdr]",
        "dynamic expression[fdr]"
    ).alias("fantomcat_transcript_info")
)


display(gencode_v49_v_fantomcat_stringent_id_mapping)

lncrna_transcript_id,id_correspondence_flag,Transcript_classification_codes,FANTOM-CAT_transcript_lv4_stringent,gene class,fantomcat_transcript_info
str,str,str,str,str,list[struct[7]]
"""ENST00000596617.5""","""2_same_id_diff_version""","""=""","""ENST00000596617.1""","""protein coding mRNAs""","[{""ENSG00000269533.1"",""promoter"",null,null,null,null,2.2200e-12}]"
…,…,…,…,…,…
"""ENST00000395862.7""","""2_same_id_diff_version""","""=""","""ENST00000395862.3""","""protein coding mRNAs""","[{""ENSG00000155097.7"",""promoter"",310.7,1884.1,""0.0184"",null,5.5300e-14}]"


In [10]:
common_cre_fantomcat_mapping = common_cre.join(
    gencode_v49_v_fantomcat_stringent_id_mapping,
    on="lncrna_transcript_id",
    how="left"
)

common_cre_fantomcat_mapping.write_parquet(
    "../../out/transcript_classification/divergent_transcript/pc_common_cre_lncRNA_all/up5p_fantabio_pc_common_cre_lncRNA_fantomcat_mapping.parquet"
)

display(common_cre_fantomcat_mapping)

cre_id,cre_name,fantabio_cre_class,cre_chrom,cre_chrom_start,cre_chrom_end,directionality,pc_transcript_distance,screen_ccres,fantom5_enhancers,pc_transcript_id,pc_HN2,pc_transcript_strand,pc_transcript_info,lncrna_transcript_distance,lncrna_transcript_id,lncrna_HN2,lncrna_transcript_strand,lncrna_transcript_info,strand_type,pc_lncrna_HN2,id_correspondence_flag,Transcript_classification_codes,FANTOM-CAT_transcript_lv4_stringent,gene class,fantomcat_transcript_info
str,str,str,str,i64,i64,f64,i64,list[struct[2]],list[struct[1]],str,i64,str,struct[5],i64,str,i64,str,struct[5],str,i64,str,str,str,str,list[struct[7]]
"""FCHS_373931""","""cp4@ZFAND2A,cp1@ZFAND2A-DT""","""PLA""","""chr7""",1160610,1160913,-0.984546,432,"[{""EH38E3752326"",""PLS""}]",null,"""ENST00000401903.6""",317,"""-""","{""ENSG00000178381.14"",""ZFAND2A"",""protein_coding"",null,[""not_organism_supported"", ""basic"", … ""CCDS""]}",-178,"""ENST00000662926.1""",130,"""+""","{""ENSG00000229043.4"",""ZFAND2A-DT"",""lncRNA"",""URS0000EF6DEC"",[""TAGENE""]}","""opposite strand""",447,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_212037""","""cp2@KLK6""","""PLA""","""chr19""",50968585,50969315,-0.997276,389,"[{""EH38E3314744"",""PLS""}, {""EH38E3314745"",""pELS""}]",null,"""ENST00000594641.1""",44,"""-""","{""ENSG00000167755.16"",""KLK6"",""protein_coding"",null,[""alternative_5_UTR"", ""upstream_ATG"", … ""CCDS""]}",-392,"""ENST00000671133.1""",50,"""+""","{""ENSG00000267879.2"",null,""lncRNA"",""URS0000EF307B"",[""basic"", ""TAGENE""]}","""opposite strand""",94,null,null,null,null,null


&nbsp;

&nbsp;

## 1-7: Classification

- 以下のタイプに分けてファイルを保存しておく

1. protein-coding transcript と lncRNA が 共通するシスエレメントを持ち、かつ 同じ strand から転写されている (5-1: Same Strand)
2. protein-coding transcript と lncRNA が共通するシスエレメント (PLA) を持ち、かつ 反対方向の strand から転写されている (5-2: Divergent Transcript (stringent selection criteria))
3. protein-coding transcript と lncRNA が共通するシスエレメント (ELA) を持ち、かつ 反対方向の strand から転写されている (5-3: Divergent Transcript (but Enhancer level activity))

&nbsp;

## 1-7-1: Same strand

In [11]:
common_cre_same_strand = common_cre_fantomcat_mapping.filter(
    pl.col("strand_type") == "same strand"
).sort(
    by="pc_lncrna_HN2",
    descending=True
)

common_cre_same_strand.write_parquet(
    "../../out/transcript_classification/divergent_transcript/pc_common_cre_lncRNA_samestrand/up5p_fantabio_pc_common_cre_lncRNA_same_strand.parquet"
)

pl.Config.set_tbl_rows(10)
print(f"unique CRE count: {common_cre_same_strand.select(['cre_id']).n_unique()}")
display(common_cre_same_strand)

unique CRE count: 130


cre_id,cre_name,fantabio_cre_class,cre_chrom,cre_chrom_start,cre_chrom_end,directionality,pc_transcript_distance,screen_ccres,fantom5_enhancers,pc_transcript_id,pc_HN2,pc_transcript_strand,pc_transcript_info,lncrna_transcript_distance,lncrna_transcript_id,lncrna_HN2,lncrna_transcript_strand,lncrna_transcript_info,strand_type,pc_lncrna_HN2,id_correspondence_flag,Transcript_classification_codes,FANTOM-CAT_transcript_lv4_stringent,gene class,fantomcat_transcript_info
str,str,str,str,i64,i64,f64,i64,list[struct[2]],list[struct[1]],str,i64,str,struct[5],i64,str,i64,str,struct[5],str,i64,str,str,str,str,list[struct[7]]
"""FCHS_418498""","""cp1@APTX,cp3@DNAJA1""","""PLA""","""chr9""",33024855,33025429,0.98779,0,"[{""EH38E3879938"",""PLS""}, {""EH38E3879939"",""PLS""}]",null,"""ENST00000851568.1""",271,"""+""","{""ENSG00000086061.17"",""DNAJA1"",""protein_coding"",null,[""basic"", ""TAGENE""]}",422,"""ENST00000813540.1""",239,"""+""","{""ENSG00000305860.1"",null,""lncRNA"",""URS0002A6B443"",[""basic"", ""Ensembl_canonical"", … ""TAGENE""]}","""same strand""",510,null,null,null,null,null
"""FCHS_418499""","""FCHS_418499""","""PLA""","""chr9""",33025623,33025964,0.979695,-348,"[{""EH38E3879941"",""pELS""}]",null,"""ENST00000851568.1""",271,"""+""","{""ENSG00000086061.17"",""DNAJA1"",""protein_coding"",null,[""basic"", ""TAGENE""]}",0,"""ENST00000813540.1""",239,"""+""","{""ENSG00000305860.1"",null,""lncRNA"",""URS0002A6B443"",[""basic"", ""Ensembl_canonical"", … ""TAGENE""]}","""same strand""",510,null,null,null,null,null
"""FCHS_418498""","""cp1@APTX,cp3@DNAJA1""","""PLA""","""chr9""",33024855,33025429,0.98779,0,"[{""EH38E3879938"",""PLS""}, {""EH38E3879939"",""PLS""}]",null,"""ENST00000851563.1""",242,"""+""","{""ENSG00000086061.17"",""DNAJA1"",""protein_coding"",null,[""basic"", ""GENCODE_Primary"", … ""CCDS""]}",422,"""ENST00000813540.1""",239,"""+""","{""ENSG00000305860.1"",null,""lncRNA"",""URS0002A6B443"",[""basic"", ""Ensembl_canonical"", … ""TAGENE""]}","""same strand""",481,null,null,null,null,null
"""FCHS_418499""","""FCHS_418499""","""PLA""","""chr9""",33025623,33025964,0.979695,-348,"[{""EH38E3879941"",""pELS""}]",null,"""ENST00000851563.1""",242,"""+""","{""ENSG00000086061.17"",""DNAJA1"",""protein_coding"",null,[""basic"", ""GENCODE_Primary"", … ""CCDS""]}",0,"""ENST00000813540.1""",239,"""+""","{""ENSG00000305860.1"",null,""lncRNA"",""URS0002A6B443"",[""basic"", ""Ensembl_canonical"", … ""TAGENE""]}","""same strand""",481,null,null,null,null,null
"""FCHS_418499""","""FCHS_418499""","""PLA""","""chr9""",33025623,33025964,0.979695,-154,"[{""EH38E3879941"",""pELS""}]",null,"""ENST00000959511.1""",240,"""+""","{""ENSG00000086061.17"",""DNAJA1"",""protein_coding"",null,[""basic"", ""TAGENE"", … ""CCDS""]}",0,"""ENST00000813540.1""",239,"""+""","{""ENSG00000305860.1"",null,""lncRNA"",""URS0002A6B443"",[""basic"", ""Ensembl_canonical"", … ""TAGENE""]}","""same strand""",479,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_183200""","""cp1@SNHG20""","""ELA""","""chr17""",77088316,77088624,0.108911,125,"[{""EH38E1887646"",""PLS""}, {""EH38E3245221"",""pELS""}]",null,"""ENST00000392476.6""",44,"""+""","{""ENSG00000129657.17"",""SEC14L1"",""protein_coding"",null,[""basic"", ""appris_principal_1""]}",88,"""ENST00000648984.1""",57,"""+""","{""ENSG00000234912.12"",""SNHG20"",""lncRNA"",""URS0000DB9F6E"",[]}","""same strand""",101,null,null,null,null,null
"""FCHS_39527""","""FCHS_39527""","""ELA""","""chr1""",231528783,231529252,0.565217,180,"[{""EH38E1431464"",""pELS""}, {""EH38E2873392"",""pELS""}]",null,"""ENST00000653908.1""",48,"""-""","{""ENSG00000287856.1"",null,""protein_coding"",null,[""RNA_Seq_supported_only"", ""basic"", … ""readthrough_transcript""]}",451,"""ENST00000659999.1""",47,"""-""","{""ENSG00000233461.6"",""TSNAX-DT"",""lncRNA"",""URS0000EED7F8"",[""RNA_Seq_supported_only""]}","""same strand""",95,null,null,null,null,null
"""FCHS_39525""","""cp3@CYDAER""","""PLA""","""chr1""",2315

&nbsp;

&nbsp;

## 1-7-2: Divergent Transcript (stringent selection criteria)

- 以下の条件を満たすペアを選抜

&nbsp;

1. protein-coding genes and lncRNA transcripts are located on the oppsite strand
2. `Promoter Level Activity (PLA)` entry

- Flag (not a filter): whether SCREEN also supports promoter classification (`PLS`) for a cCRE overlapping the fanta.bio CRE. This is stored in `screen_promoter_support`.
- Flag (not a filter): whether a GENCODE/Ensembl promoter window maps to this fanta.bio CRE in `ensembl_promoter_intersect`. This is stored in `ensembl_promoter_support`.
  - related notebook: [../02_fantabio/02_4_ensembl_promoter_intersect.ipynb](../02_fantabio/02_4_ensembl_promoter_intersect.ipynb)

In [12]:
common_cre_divergent = common_cre_fantomcat_mapping.filter(
    (pl.col("strand_type") == "opposite strand") & # condition 1
    (pl.col("fantabio_cre_class") == "PLA") # condition 2
).with_columns(

    # flag1: SCREEN also supports promoter (PLS)
    pl.when(
        pl.col("screen_ccres")
        .list.eval(pl.element().struct.field("screen_ccre_type") == "PLS")
        .list.any()
        .fill_null(False)
    ).then(
        pl.lit("supported by SCREEN")
    ).otherwise(
        pl.lit("not supported by SCREEN")
    ).alias("screen_promoter_support"),


    # flag2: CRE is the representative fanta.bio overlap of a GENCODE/Ensembl promoter window
    pl.when(
        pl.col("cre_id").is_in(pl.lit(ensembl_promoter_intersect.get_column("cre_id")).implode())
        .fill_null(False)
    ).then(
        pl.lit("supported by Ensembl/GENCODE promoter window")
    ).otherwise(
        pl.lit("not supported by Ensembl/GENCODE promoter window")
    ).alias("ensembl_promoter_window_support"),


    # flag3: FANTOM-CAT support
    pl.when(
        pl.col("gene class") == "lncRNA, divergent"
    ).then(
        pl.lit("supported by FANTOM-CAT (stringent)")
    ).otherwise(
        pl.lit("not supported by FANTOM-CAT (stringent)")
    ).alias("fantomcat_stringent_support")
).sort(
    by="pc_lncrna_HN2",
    descending=True
)

common_cre_divergent.write_parquet(
    "../../out/transcript_classification/divergent_transcript/pc_common_cre_lncRNA_divergent/up5p_fantabio_pc_common_cre_lncRNA_divergent.parquet"
)

pl.Config.set_tbl_rows(10)
print(f"unique divergent CRE count: {common_cre_divergent.select(['cre_id']).n_unique()}")
print(f"unique protein-coding transcript count: {common_cre_divergent.select(['pc_transcript_id']).n_unique()}")
print(f"unique lncRNA transcript count: {common_cre_divergent.select(['lncrna_transcript_id']).n_unique()}")
print(f"unique lncRNA - CRE pair count: {common_cre_divergent.select(['cre_id', 'lncrna_transcript_id']).n_unique()}")
display(common_cre_divergent)

unique divergent CRE count: 598
unique protein-coding transcript count: 895
unique lncRNA transcript count: 485
unique lncRNA - CRE pair count: 884


cre_id,cre_name,fantabio_cre_class,cre_chrom,cre_chrom_start,cre_chrom_end,directionality,pc_transcript_distance,screen_ccres,fantom5_enhancers,pc_transcript_id,pc_HN2,pc_transcript_strand,pc_transcript_info,lncrna_transcript_distance,lncrna_transcript_id,lncrna_HN2,lncrna_transcript_strand,lncrna_transcript_info,strand_type,pc_lncrna_HN2,id_correspondence_flag,Transcript_classification_codes,FANTOM-CAT_transcript_lv4_stringent,gene class,fantomcat_transcript_info,screen_promoter_support,ensembl_promoter_window_support,fantomcat_stringent_support
str,str,str,str,i64,i64,f64,i64,list[struct[2]],list[struct[1]],str,i64,str,struct[5],i64,str,i64,str,struct[5],str,i64,str,str,str,str,list[struct[7]],str,str,str
"""FCHS_110516""","""cp7@HSPH1""","""PLA""","""chr13""",31161745,31162676,-0.981672,0,"[{""EH38E3059049"",""pELS""}, {""EH38E3059050"",""PLS""}, … {""EH38E3059052"",""pELS""}]",null,"""ENST00000954002.1""",263,"""-""","{""ENSG00000120694.21"",""HSPH1"",""protein_coding"",null,[""basic"", ""TAGENE"", ""CCDS""]}",0,"""ENST00000729765.1""",295,"""+""","{""ENSG00000295391.1"",null,""lncRNA"",""URS0002A422B0"",[""basic"", ""TAGENE""]}","""opposite strand""",558,"""4_equal_fantom_native""","""=""","""FTMT25200000861.1""","""uncertain coding""","[{""CATG00000014230.1"",""promoter"",138.6,730.6,null,null,1.3800e-10}]","""supported by SCREEN""","""supported by Ensembl/GENCODE promoter window""","""not supported by FANTOM-CAT (stringent)"""
"""FCHS_329552""","""cp12@GPBP1""","""PLA""","""chr5""",57173435,57174283,0.917886,0,"[{""EH38E3637789"",""PLS""}, {""EH38E3637790"",""PLS""}, … {""EH38E4368312"",""pELS""}]",null,"""ENST00000935651.1""",300,"""+""","{""ENSG00000062194.17"",""GPBP1"",""protein_coding"",null,[""basic"", ""GENCODE_Primary"", … ""CCDS""]}",0,"""ENST00000688485.3""",255,"""-""","{""ENSG00000289152.3"",null,""lncRNA"",""URS0002A1B76A"",[""TAGENE""]}","""opposite strand""",555,null,null,null,null,null,"""supported by SCREEN""","""supported by Ensembl/GENCODE promoter window""","""not supported by FANTOM-CAT (stringent)"""
"""FCHS_354486""","""cp1@HSPA1A,cp1@HSPA1B""","""PLA""","""chr6""",31827058,31827936,0.997399,0,"[{""EH38E3700980"",""pELS""}, {""EH38E3700981"",""pELS""}, {""EH38E3700982"",""PLS""}]",null,"""ENST00000375650.5""",286,"""+""","{""ENSG00000204388.7"",""HSPA1B"",""protein_coding"",null,[""basic"", ""Ensembl_canonical"", … ""CCDS""]}",0,"""ENST00000700791.2""",251,"""-""","{""ENSG00000289829.2"",null,""lncRNA"",""URS0002A24591"",[""basic"", ""Ensembl_canonical"", … ""TAGENE""]}","""opposite strand""",537,null,null,null,null,null,"""supported by SCREEN""","""supported by Ensembl/GENCODE promoter window""","""not supported by FANTOM-CAT (stringent)"""
"""FCHS_354485""","""FCHS_354485""","""PLA""","""chr6""",31826812,31827243,-0.869084,495,"[{""EH38E3700980"",""pELS""}]",null,"""ENST00000375650.5""",286,"""+""","{""ENSG00000204388.7"",""HSPA1B"",""protein_coding"",null,[""basic"", ""Ensembl_canonical"", … ""CCDS""]}",-200,"""ENST00000700791.2""",251,"""-""","{""ENSG00000289829.2"",null,""lncRNA"",""URS0002A24591"",[""basic"", ""Ensembl_canonical"", … ""TAGENE""]}","""opposite strand""",537,null,null,null,null,null,"""not supported by SCREEN""","""not supported by Ensembl/GENCODE promoter window""","""not supported by FANTOM-CAT (stringent)"""
"""FCHS_78719""","""cp2@CHORDC1,cp3@DISC1FP1""","""PLA""","""chr11""",90222852,90223461,-0.954578,0,"[{""EH38E2978133"",""pELS""}, {""EH38E2978134"",""PLS""}, {""EH38E2978135"",""pELS""}]",null,"""ENST00000907759.1""",295,"""-""","{""ENSG00000110172.13"",""CHORDC1"",""protein_coding"",null,[""basic"", ""TAGENE""]}",0,"""ENST00000751874.1""",238,"""+""","{""ENSG00000297932.1"",null,""lncRNA"",""URS0002A2FD75"",[""basic"", ""Ensembl_canonical"", … ""TAGENE""]}","""opposite strand""",533,"""4_equal_fantom_native""","""=""","""MICT00000065637.1""","""lncRNA, divergent""","[{""CATG00000003338.1"",""promoter"",205.5,null,null,7.8809e-44,0.000004}]","""supported by SCREEN""","

&nbsp;

&nbsp;

## 1-7-3: Divergent Transcript (but Enhancer level activity)

- 既に`common_cre_divergent`でペアが定義されているものも含まれていることに注意
- (e.g., FCHS_361737 の他に、FCHS_361738 のペアも近傍にあるが、FCHS_361738 のペアは ELA のため、`common_cre_divergent`には含まれていない)

In [13]:
common_cre_divergent_ela = common_cre_fantomcat_mapping.filter(
    (pl.col("strand_type") == "opposite strand") &
    (pl.col("fantabio_cre_class") == "ELA")
).sort(
    by="cre_id",
    descending=True
).with_columns(
    pl.when(
        # flag: SCREEN also supports promoter (PLS)
        pl.col("screen_ccres")
        .list.eval(pl.element().struct.field("screen_ccre_type") == "pELS")
        .list.any()
        .fill_null(False)
    ).then(
        pl.lit("supported by SCREEN")
    ).otherwise(
        pl.lit("not supported by SCREEN")
    ).alias("screen_pELS_support"),


    # flag: CRE is the representative fanta.bio overlap of a GENCODE/Ensembl promoter window
    pl.when(
        pl.col("cre_id").is_in(pl.lit(ensembl_promoter_intersect.get_column("cre_id")).implode())
        .fill_null(False)
    ).then(
        pl.lit("supported by Ensembl/GENCODE promoter window")
    ).otherwise(
        pl.lit("not supported by Ensembl/GENCODE promoter window")
    ).alias("ensembl_promoter_window_support"),


    # flag: FANTOM-CAT support
    pl.when(
        pl.col("gene class") == "lncRNA, divergent"
    ).then(
        pl.lit("supported by FANTOM-CAT (stringent)")
    ).otherwise(
        pl.lit("not supported by FANTOM-CAT (stringent)")
    ).alias("fantomcat_stringent_support")
)

pl.Config.set_tbl_rows(10)
print(f"unique divergent CRE count: {common_cre_divergent_ela.select(['cre_id']).n_unique()}")
print(f"unique protein-coding transcript count: {common_cre_divergent_ela.select(['pc_transcript_id']).n_unique()}")
print(f"unique lncRNA transcript count: {common_cre_divergent_ela.select(['lncrna_transcript_id']).n_unique()}")
display(common_cre_divergent_ela)

unique divergent CRE count: 376
unique protein-coding transcript count: 535
unique lncRNA transcript count: 319


cre_id,cre_name,fantabio_cre_class,cre_chrom,cre_chrom_start,cre_chrom_end,directionality,pc_transcript_distance,screen_ccres,fantom5_enhancers,pc_transcript_id,pc_HN2,pc_transcript_strand,pc_transcript_info,lncrna_transcript_distance,lncrna_transcript_id,lncrna_HN2,lncrna_transcript_strand,lncrna_transcript_info,strand_type,pc_lncrna_HN2,id_correspondence_flag,Transcript_classification_codes,FANTOM-CAT_transcript_lv4_stringent,gene class,fantomcat_transcript_info,screen_pELS_support,ensembl_promoter_window_support,fantomcat_stringent_support
str,str,str,str,i64,i64,f64,i64,list[struct[2]],list[struct[1]],str,i64,str,struct[5],i64,str,i64,str,struct[5],str,i64,str,str,str,str,list[struct[7]],str,str,str
"""FCHS_97992""","""cp1@LLPH""","""ELA""","""chr12""",66131016,66131140,-0.666667,266,"[{""EH38E1620877"",""pELS""}]",null,"""ENST00000266604.7""",84,"""-""","{""ENSG00000139233.8"",""LLPH"",""protein_coding"",null,[""basic"", ""Ensembl_canonical"", … ""CCDS""]}",-265,"""ENST00000510317.2""",93,"""+""","{""ENSG00000239335.4"",""LLPH-DT"",""lncRNA"",""URS000004F0C7"",[""basic"", ""Ensembl_canonical"", … ""exp_conf""]}","""opposite strand""",177,"""1_exact_version""","""=""","""ENST00000510317.2""","""lncRNA, divergent""","[{""ENSG00000239335.4"",""promoter"",null,null,null,null,2.0500e-11}]","""supported by SCREEN""","""not supported by Ensembl/GENCODE promoter window""","""supported by FANTOM-CAT (stringent)"""
"""FCHS_96117""","""cp7@PA2G4""","""ELA""","""chr12""",56103854,56104130,0.529412,415,"[{""EH38E4064749"",""pELS""}]",null,"""ENST00000870200.1""",104,"""+""","{""ENSG00000170515.15"",""PA2G4"",""protein_coding"",null,[""basic"", ""TAGENE"", … ""CCDS""]}",-276,"""ENST00000732893.1""",47,"""-""","{""ENSG00000295820.1"",null,""lncRNA"",""URS0002A6942B"",[""basic"", ""Ensembl_canonical"", … ""TAGENE""]}","""opposite strand""",151,null,null,null,null,null,"""supported by SCREEN""","""not supported by Ensembl/GENCODE promoter window""","""not supported by FANTOM-CAT (stringent)"""
"""FCHS_9380""","""FCHS_9380""","""ELA""","""chr1""",31575995,31576375,0.6,140,"[{""EH38E2800361"",""PLS""}, {""EH38E3960378"",""pELS""}]",null,"""ENST00000861777.1""",57,"""+""","{""ENSG00000142910.17"",""TINAGL1"",""protein_coding"",null,[""basic"", ""TAGENE""]}",348,"""ENST00000435872.2""",49,"""-""","{""ENSG00000229167.3"",null,""lncRNA"",""URS0002A35F88"",[""basic"", ""TAGENE"", ""exp_conf""]}","""opposite strand""",106,null,null,null,null,null,"""supported by SCREEN""","""not supported by Ensembl/GENCODE promoter window""","""not supported by FANTOM-CAT (stringent)"""
"""FCHS_93632""","""FCHS_93632""","""ELA""","""chr12""",49264487,49264615,1.0,465,"[{""EH38E1609971"",""PLS""}, {""EH38E3015429"",""pELS""}]",null,"""ENST00000931111.1""",49,"""+""","{""ENSG00000167553.18"",""TUBA1C"",""protein_coding"",null,[""basic"", ""GENCODE_Primary"", ""TAGENE""]}",-168,"""ENST00000549023.3""",55,"""-""","{""ENSG00000258101.3"",null,""lncRNA"",""URS0002A64B3A"",[""basic"", ""Ensembl_canonical"", … ""TAGENE""]}","""opposite strand""",104,"""3_same_gene_diff_transcript""","""c""","""FTMT24600002200.1""","""lncRNA, divergent""","[{""ENSG00000258101.1"",""promoter"",null,null,null,null,0.000345}]","""supported by SCREEN""","""not supported by Ensembl/GENCODE promoter window""","""supported by FANTOM-CAT (stringent)"""
"""FCHS_93632""","""FCHS_93632""","""ELA""","""chr12""",49264487,49264615,1.0,465,"[{""EH38E1609971"",""PLS""}, {""EH38E3015429"",""pELS""}]",null,"""ENST00000931111.1""",49,"""+""","{""ENSG00000167553.18"",""TUBA1C"",""protein_coding"",null,[""basic"", ""GENCODE_Primary"", ""TAGENE""]}",-168,"""ENST00000549023.3""",55,"""-""","{""ENSG00000258101.3"",null,""lncRNA"",""URS0002A64B3A"",[""basic"", ""Ensembl_canonical"", … ""TAGENE""]}","""opposite strand""",104,"""2_same_id_diff_version""","""=""","""ENST00000549023.1""","""lncRNA, divergent""","[{""ENSG00000258101.1"",""promoter"",null,null,null,null,0.000345}]","""supported by SCREEN""","""not sup

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

# 2. Downregulated Transcripts (down5p)


&nbsp;

## 2-1: down5p fantabio

- [../02_fantabio/02_3_HNscore_down_transcript_fantabio.ipynb](../02_fantabio/02_3_HNscore_down_transcript_fantabio.ipynb)で処理した CRE との統合したファイルを input にする


In [14]:
down5p_fantabio_cre = pl.read_parquet(
    "../../out/fantabio/parquet_file/bottom5p_transcript_fantabio_v261001.parquet"
)

pl.Config.set_tbl_rows(2)
print(f"protein-coding transcript - CRE pair count: {down5p_fantabio_cre.filter(pl.col('transcript_type') == 'protein_coding').select(['cre_id', 'transcript_id']).n_unique()}")
print(f"lncRNA transcript - CRE pair count: {down5p_fantabio_cre.filter(pl.col('transcript_type') == 'lncRNA').select(['cre_id', 'transcript_id']).n_unique()}")
display(down5p_fantabio_cre)


protein-coding transcript - CRE pair count: 38359
lncRNA transcript - CRE pair count: 8006


transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,cre_chrom,cre_chrom_start,cre_chrom_end,cre_name,tss_nearest_transcript_ids,tss_nearest_transcript_distance,transcript_distance,screen_ccres,chip_atlas_antigens,silencerdbs,fantom5_cage_peaks,fantom5_enhancers,directionality,fantabio_cre_class,itemRgb,itemRgb_name
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,str,i64,str,str,list[str],str,i64,list[struct[2]],list[struct[3]],list[struct[5]],list[struct[2]],list[struct[1]],f64,str,str,str
"""ENST00000834029.1""",69,117,220,-48,47,56,148,-9,"""EXO5-DT""","""HGNC:55647""","""lncRNA""","""ENSG00000238287.2""","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]","""-""",4,"""URS0002A1647B""","""ENSEMBL_GENCODE""",null,"""havana_homo_sapiens""","[""CLS3 project""]",1,"""FCHS_11684""","""chr1""",40508315,"""40508650""","""cp5@EXO5""","[""ENST00000925740.1""]","""0""",-63,"[{""EH38E2805480"",""PLS""}]","[{""ZNF687"",3399,[{""SRX10478566"",3399}, {""SRX10478565"",2503}]}, {""TCF7"",2966,[{""SRX10774913"",2966}, {""SRX10774914"",2622}]}, … {""NFATC1"",515,[{""SRX335071"",515}]}]","[{""HS_00004908"",""Foreskin Fibroblast Primary Cells skin02"",""Prepuce"",""Skin"",""SVM""}, {""HS_00004909"",""Foreskin Melanocyte Primary Cells skin01"",""Prepuce"",""Skin"",""SVM""}, … {""HS_00004912"",""Small Intestine"",""Small intestine"",""Intestine"",""SVM""}]","[{""hg19::chr1:40974317..40974346,-;hg_11928.1"",""hg_11928.1""}]",null,0.952663,"""ELA""","""255,202,74""","""yellow"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000549545.1""",11,265,130,-254,8,174,69,-166,"""COQ10A""","""HGNC:26515""","""retained_intron""","""ENSG00000135469.15""",[],"""+""",0,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,"""FCHS_96252""","""chr12""",56268140,"""56268446""","""cp6@COQ10A""","[""ENST00000549545.1""]","""-23""",-23,"[{""EH38E4064794"",""pELS""}]",null,"[{""HS_00173371"",""H1 BMP4 Derived Mesendoderm Cultured Cells"",""Mesendoderm cell"",""Others"",""SVM""}, {""HS_02053667"",""Thyroid Gland"",""Thyroid gland"",""Others"",""gkmSVM""}]",null,null,-0.368421,"""ELA""","""255,202,74""","""yellow"""


&nbsp;

## 2-2: Select Columns


In [15]:
column_list = [
    "cre_id",
    "cre_name",
    "fantabio_cre_class",
    "cre_chrom",
    "cre_chrom_start",
    "cre_chrom_end",
    "directionality",
    "transcript_distance",
    "screen_ccres",
    "fantom5_enhancers",
    "transcript_id",
    "HN2",
    "gene_id",
    "gene_symbol",
    "transcript_type",
    "RNACentral_ID",
    "strand",
    "tags",
]


&nbsp;

## 2-3: Protein-coding Transcript

- CRE 側に付与される情報として加工する


In [16]:
pc_transcript = down5p_fantabio_cre.filter(
    pl.col("transcript_type") == "protein_coding"
).select(
    column_list
).unique(
    subset=["cre_id", "transcript_id"]
).with_columns(
    # Pack transcript-side fields into a struct; row grain (CRE x transcript) stays the same
    pl.struct(
        "gene_id",
        "gene_symbol",
        "transcript_type",
        "RNACentral_ID",
        "tags",
    ).alias("pc_transcript_info")
).drop(
    "gene_id",
    "gene_symbol",
    "transcript_type",
    "RNACentral_ID",
    "tags"
).rename(
    {
        "transcript_id": "pc_transcript_id",
        "strand": "pc_transcript_strand",
        "transcript_distance": "pc_transcript_distance",
        "HN2": "pc_HN2"
    }
).sort(
    "pc_HN2",
    descending=True
)


print(f"unique protein-coding transcript count: {pc_transcript.select(pl.col('pc_transcript_id')).n_unique()}")
display(pc_transcript)


unique protein-coding transcript count: 12063


cre_id,cre_name,fantabio_cre_class,cre_chrom,cre_chrom_start,cre_chrom_end,directionality,pc_transcript_distance,screen_ccres,fantom5_enhancers,pc_transcript_id,pc_HN2,pc_transcript_strand,pc_transcript_info
str,str,str,str,i64,str,f64,i64,list[struct[2]],list[struct[1]],str,i64,str,struct[5]
"""FCHS_129272""","""cp2@CPSF2""","""PLA""","""chr14""",92121987,"""92122279""",0.324937,-27,"[{""EH38E3111458"",""PLS""}, {""EH38E3111459"",""pELS""}]",null,"""ENST00000932646.1""",-48,"""+""","{""ENSG00000165934.14"",""CPSF2"",""protein_coding"",null,[""basic"", ""TAGENE"", … ""CCDS""]}"
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_76429""","""FCHS_76429""","""ELA""","""chr11""",72813825,"""72814031""",0.428571,401,"[{""EH38E2971311"",""PLS""}]","[{[""chr11:72812651-72813869""]}]","""ENST00000540567.1""",-251,"""+""","{""ENSG00000168010.12"",""ATG16L2"",""protein_coding"",null,[""mRNA_start_NF"", ""cds_start_NF""]}"


&nbsp;

## 2-4: lncRNA transcript


In [17]:
lncrna_transcript = down5p_fantabio_cre.filter(
    pl.col("transcript_type") == "lncRNA"
).select(
    column_list
).unique(
    subset=["cre_id", "transcript_id"]
).with_columns(
    # Pack transcript-side fields into a struct; row grain (CRE x transcript) stays the same
    pl.struct(
        "gene_id",
        "gene_symbol",
        "transcript_type",
        "RNACentral_ID",
        "tags",
    ).alias("lncrna_transcript_info")
).drop(
    "gene_id",
    "gene_symbol",
    "transcript_type",
    "RNACentral_ID",
    "tags"
).rename(
    {
        "transcript_id": "lncrna_transcript_id",
        "strand": "lncrna_transcript_strand",
        "transcript_distance": "lncrna_transcript_distance",
        "HN2": "lncrna_HN2"
    }
).sort(
    "lncrna_HN2",
    descending=True
)

print(f"unique lncRNA transcript count: {lncrna_transcript.select(pl.col('lncrna_transcript_id')).n_unique()}")
display(lncrna_transcript)


unique lncRNA transcript count: 2653


cre_id,cre_name,fantabio_cre_class,cre_chrom,cre_chrom_start,cre_chrom_end,directionality,lncrna_transcript_distance,screen_ccres,fantom5_enhancers,lncrna_transcript_id,lncrna_HN2,lncrna_transcript_strand,lncrna_transcript_info
str,str,str,str,i64,str,f64,i64,list[struct[2]],list[struct[1]],str,i64,str,struct[5]
"""FCHS_124314""","""cp40@PRKCH,cp4@PRKCH-AS1""","""PLA""","""chr14""",61322620,"""61323244""",-0.889414,0,"[{""EH38E1719382"",""PLS""}, {""EH38E3098356"",""PLS""}, {""EH38E3098357"",""pELS""}]",null,"""ENST00000500036.2""",-48,"""-""","{""ENSG00000247287.4"",""PRKCH-AS1"",""lncRNA"",""URS00007DFED0"",[""basic""]}"
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_304130""","""cp4@MUC20-OT1,cp1@SDHAP2""","""PLA""","""chr3""",195657735,"""195658218""",0.905308,0,"[{""EH38E3563576"",""PLS""}, {""EH38E4323230"",""pELS""}]",null,"""ENST00000446521.2""",-215,"""+""","{""ENSG00000242086.9"",""MUC20-OT1"",""lncRNA"",""URS0002A4231D"",[""TAGENE"", ""readthrough_transcript""]}"


&nbsp;

## 2-5: Common CRE

- 作成したデータフレームを `join` することで共通する CRE を特定する
- ここで調べることができるのは、protein-coding transcript と lncRNA transcript 両方が `downregulated` として分類されているペア
- 簡易的な指標として、protein-coding transcript と lncRNA transcript の HN-score の値の和を計算しておく


In [18]:
common_cre = pc_transcript.join(
    lncrna_transcript,
    on=[
        "cre_id",
        "cre_name",
        "fantabio_cre_class",
        "cre_chrom",
        "cre_chrom_start",
        "cre_chrom_end",
        "screen_ccres",
        "fantom5_enhancers",
        "directionality"
    ],
    how="inner",
    coalesce=True,
    nulls_equal=True # これを指定しないと、strand が `null` のエントリーが `join` されない
).with_columns(
    pl.when(
        pl.col("pc_transcript_strand") == pl.col("lncrna_transcript_strand")
    ).then(
        pl.lit("same strand")
    ).otherwise(
        pl.when(
            # explicitly check for strand mismatch
            pl.col("pc_transcript_strand") != pl.col("lncrna_transcript_strand")
        ).then(
            pl.lit("opposite strand")
        ).otherwise(
            pl.lit("unknown strand")
        )
    ).alias("strand_type")
).with_columns(
    (pl.col("pc_HN2") + pl.col("lncrna_HN2")).alias("pc_lncrna_HN2") # 簡易的な指標として、HN2 の和を計算
).cast(
    # なぜか Int64 になっていないので、Int64 に変換
    {
        "cre_chrom_end": pl.Int64
    }
)

print(f"unique CRE count (including PLA and ELA): {common_cre.select(['cre_id']).n_unique()}")
display(common_cre)


unique CRE count (including PLA and ELA): 1536


cre_id,cre_name,fantabio_cre_class,cre_chrom,cre_chrom_start,cre_chrom_end,directionality,pc_transcript_distance,screen_ccres,fantom5_enhancers,pc_transcript_id,pc_HN2,pc_transcript_strand,pc_transcript_info,lncrna_transcript_distance,lncrna_transcript_id,lncrna_HN2,lncrna_transcript_strand,lncrna_transcript_info,strand_type,pc_lncrna_HN2
str,str,str,str,i64,i64,f64,i64,list[struct[2]],list[struct[1]],str,i64,str,struct[5],i64,str,i64,str,struct[5],str,i64
"""FCHS_365488""","""FCHS_365488""","""ELA""","""chr6""",119350037,119350562,-0.672727,276,"[{""EH38E2498136"",""PLS""}, {""EH38E3729392"",""pELS""}]",null,"""ENST00000368468.4""",-48,"""-""","{""ENSG00000111885.8"",""MAN1A1"",""protein_coding"",null,[""basic"", ""Ensembl_canonical"", … ""CCDS""]}",-138,"""ENST00000689073.3""",-60,"""+""","{""ENSG00000287100.4"",null,""lncRNA"",""URS0002A17A25"",[""TAGENE""]}","""opposite strand""",-108
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_76429""","""FCHS_76429""","""ELA""","""chr11""",72813825,72814031,0.428571,401,"[{""EH38E2971311"",""PLS""}]","[{[""chr11:72812651-72813869""]}]","""ENST00000540567.1""",-251,"""+""","{""ENSG00000168010.12"",""ATG16L2"",""protein_coding"",null,[""mRNA_start_NF"", ""cds_start_NF""]}",-41,"""ENST00000702661.1""",-78,"""-""","{""ENSG00000285693.3"",null,""lncRNA"",""URS00025E7D67"",[""basic"", ""TAGENE""]}","""opposite strand""",-329


&nbsp;

## 2-6: Integration Resources

- 下流の解析で flag などに使用するデータをここで読み込んでおく

&nbsp;

1. ensembl promoter window (GENCODE release 48) - fanta.bio mapping file
2. GENCODE release 49 vs FANTOM-CAT (stringent) ID mapping file


&nbsp;

## 2-6-1: ensembl promoter window (GENCODE release 48) - fanta.bio mapping file


In [19]:
ensembl_promoter_intersect = pl.read_csv(
    "../../out/fantabio/ensembl_promoter/gencode.v48.promoter_windows_v260908_intersecting_human-CREv1.2.1.hg38.cre-peaks_v261003_one_per_promoter.tsv",
    separator="\t"
)

display(ensembl_promoter_intersect)


chromosome_ensembl,start_ensembl,end_ensembl,ensembl_promoter_accession,strand,chromosome_fantabio,start_fantabio,end_fantabio,cre_id,cre_name,intersect_length,fantabio_cre_overlap_frac_of_ensembl_promoter_window,directionality,fantabio_cre_class
str,i64,i64,str,str,str,i64,i64,str,str,i64,f64,f64,str
"""chrX""",100636806,100637806,"""ENSG00000000003_promoter_window""","""-""","""chrX""",100636532,100637120,"""FCHS_442038""","""cp4@TSPAN6""",314,0.314,-0.998632,"""PLA"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chr19""",1274529,1275529,"""ENSG00000310576_promoter_window""","""+""","""chr19""",1275363,1275715,"""FCHS_196119""","""cp3@FAM174C""",166,0.166,0.991043,"""PLA"""


&nbsp;

## 2-6-2. GENCODE release 49 vs FANTOM-CAT (stringent) ID mapping file


In [21]:
gencode_v49_v_fantomcat_stringent_id_mapping = pl.read_csv(
    "../../out/GENCODE_FANTOMCAT_comparison/GENCODE_v49_FANTOMCAT_lv4_stringent_comparison_v261010.tsv",
    separator="\t"
).rename(
    {
        "GENCODE_transcript": "lncrna_transcript_id"
    }
).group_by(
    "lncrna_transcript_id",
    "id_correspondence_flag",
    "Transcript_classification_codes",
    "FANTOM-CAT_transcript_lv4_stringent",
    "gene class"
).agg(
    pl.struct(
        "FANTOM-CAT_gene_lv4_stringent",
        "DHS support",
        "TIR conservation[RS score]",
        "exon conservation[RS score]",
        "trait vs cell enrichment[fdr]",
        "eQTL mRNA coexpression[fdr]",
        "dynamic expression[fdr]"
    ).alias("fantomcat_transcript_info")
)


display(gencode_v49_v_fantomcat_stringent_id_mapping)


lncrna_transcript_id,id_correspondence_flag,Transcript_classification_codes,FANTOM-CAT_transcript_lv4_stringent,gene class,fantomcat_transcript_info
str,str,str,str,str,list[struct[7]]
"""ENST00000439775.1""","""6_other""","""=""","""ENST00000439775.1""","""protein coding mRNAs""","[{""ENSG00000100028.7"",""promoter"",122.7,836.7,null,null,1.7100e-55}]"
…,…,…,…,…,…
"""ENST00000422287.6""","""2_same_id_diff_version""","""=""","""ENST00000422287.2""","""protein coding mRNAs""","[{""ENSG00000145414.4"",""promoter"",null,null,null,null,9.3000e-23}]"


In [22]:
common_cre_fantomcat_mapping = common_cre.join(
    gencode_v49_v_fantomcat_stringent_id_mapping,
    on="lncrna_transcript_id",
    how="left"
)

common_cre_fantomcat_mapping.write_parquet(
    "../../out/transcript_classification/divergent_transcript/pc_common_cre_lncRNA_all/down5p_fantabio_pc_common_cre_lncRNA_fantomcat_mapping.parquet"
)

display(common_cre_fantomcat_mapping)


cre_id,cre_name,fantabio_cre_class,cre_chrom,cre_chrom_start,cre_chrom_end,directionality,pc_transcript_distance,screen_ccres,fantom5_enhancers,pc_transcript_id,pc_HN2,pc_transcript_strand,pc_transcript_info,lncrna_transcript_distance,lncrna_transcript_id,lncrna_HN2,lncrna_transcript_strand,lncrna_transcript_info,strand_type,pc_lncrna_HN2,id_correspondence_flag,Transcript_classification_codes,FANTOM-CAT_transcript_lv4_stringent,gene class,fantomcat_transcript_info
str,str,str,str,i64,i64,f64,i64,list[struct[2]],list[struct[1]],str,i64,str,struct[5],i64,str,i64,str,struct[5],str,i64,str,str,str,str,list[struct[7]]
"""FCHS_365488""","""FCHS_365488""","""ELA""","""chr6""",119350037,119350562,-0.672727,276,"[{""EH38E2498136"",""PLS""}, {""EH38E3729392"",""pELS""}]",null,"""ENST00000368468.4""",-48,"""-""","{""ENSG00000111885.8"",""MAN1A1"",""protein_coding"",null,[""basic"", ""Ensembl_canonical"", … ""CCDS""]}",-138,"""ENST00000689073.3""",-60,"""+""","{""ENSG00000287100.4"",null,""lncRNA"",""URS0002A17A25"",[""TAGENE""]}","""opposite strand""",-108,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_76429""","""FCHS_76429""","""ELA""","""chr11""",72813825,72814031,0.428571,401,"[{""EH38E2971311"",""PLS""}]","[{[""chr11:72812651-72813869""]}]","""ENST00000540567.1""",-251,"""+""","{""ENSG00000168010.12"",""ATG16L2"",""protein_coding"",null,[""mRNA_start_NF"", ""cds_start_NF""]}",-41,"""ENST00000702661.1""",-78,"""-""","{""ENSG00000285693.3"",null,""lncRNA"",""URS00025E7D67"",[""basic"", ""TAGENE""]}","""opposite strand""",-329,null,null,null,null,null


&nbsp;

## 2-7: Classification

- 以下のタイプに分けてファイルを保存しておく

1. protein-coding transcript と lncRNA が 共通するシスエレメントを持ち、かつ 同じ strand から転写されている (2-7-1: Same Strand)
2. protein-coding transcript と lncRNA が共通するシスエレメント (PLA) を持ち、かつ 反対方向の strand から転写されている (2-7-2: Divergent Transcript (stringent selection criteria))
3. protein-coding transcript と lncRNA が共通するシスエレメント (ELA) を持ち、かつ 反対方向の strand から転写されている (2-7-3: Divergent Transcript (but Enhancer level activity))


&nbsp;

## 2-7-1: Same strand


In [23]:
common_cre_same_strand = common_cre_fantomcat_mapping.filter(
    pl.col("strand_type") == "same strand"
).sort(
    by="pc_lncrna_HN2",
    descending=True
)

common_cre_same_strand.write_parquet(
    "../../out/transcript_classification/divergent_transcript/pc_common_cre_lncRNA_samestrand/down5p_fantabio_pc_common_cre_lncRNA_same_strand.parquet"
)

pl.Config.set_tbl_rows(10)
print(f"unique CRE count: {common_cre_same_strand.select(['cre_id']).n_unique()}")
display(common_cre_same_strand)


unique CRE count: 61


cre_id,cre_name,fantabio_cre_class,cre_chrom,cre_chrom_start,cre_chrom_end,directionality,pc_transcript_distance,screen_ccres,fantom5_enhancers,pc_transcript_id,pc_HN2,pc_transcript_strand,pc_transcript_info,lncrna_transcript_distance,lncrna_transcript_id,lncrna_HN2,lncrna_transcript_strand,lncrna_transcript_info,strand_type,pc_lncrna_HN2,id_correspondence_flag,Transcript_classification_codes,FANTOM-CAT_transcript_lv4_stringent,gene class,fantomcat_transcript_info
str,str,str,str,i64,i64,f64,i64,list[struct[2]],list[struct[1]],str,i64,str,struct[5],i64,str,i64,str,struct[5],str,i64,str,str,str,str,list[struct[7]]
"""FCHS_95090""","""cp4@CSAD,cp2@ZNF740""","""PLA""","""chr12""",53180291,53180891,0.653935,0,"[{""EH38E1613275"",""pELS""}, {""EH38E3018479"",""PLS""}, {""EH38E3018480"",""PLS""}]",null,"""ENST00000885696.1""",-51,"""-""","{""ENSG00000139631.21"",""CSAD"",""protein_coding"",null,[""basic"", ""GENCODE_Primary"", … ""CCDS""]}",-428,"""ENST00000739605.1""",-48,"""-""","{""ENSG00000296428.1"",null,""lncRNA"",""URS0002A51155"",[""basic"", ""Ensembl_canonical"", … ""TAGENE""]}","""same strand""",-99,null,null,null,null,null
"""FCHS_95092""","""cp9@CSAD,cp1@ZNF740""","""PLA""","""chr12""",53180749,53181273,0.239375,109,"[{""EH38E3018479"",""PLS""}, {""EH38E3018480"",""PLS""}, {""EH38E3018481"",""pELS""}]",null,"""ENST00000885696.1""",-51,"""-""","{""ENSG00000139631.21"",""CSAD"",""protein_coding"",null,[""basic"", ""GENCODE_Primary"", … ""CCDS""]}",-46,"""ENST00000739605.1""",-48,"""-""","{""ENSG00000296428.1"",null,""lncRNA"",""URS0002A51155"",[""basic"", ""Ensembl_canonical"", … ""TAGENE""]}","""same strand""",-99,null,null,null,null,null
"""FCHS_73779""","""cp1@ZNRD2-DT""","""PLA""","""chr11""",65569891,65570289,-0.544465,170,"[{""EH38E2964564"",""PLS""}, {""EH38E2964565"",""PLS""}]",null,"""ENST00000913761.1""",-49,"""+""","{""ENSG00000173465.9"",""ZNRD2"",""protein_coding"",null,[""basic"", ""GENCODE_Primary"", ""TAGENE""]}",-61,"""ENST00000726069.1""",-52,"""+""","{""ENSG00000294808.1"",null,""lncRNA"",""URS0002A38452"",[""basic"", ""Ensembl_canonical"", … ""TAGENE""]}","""same strand""",-101,"""5_c_fantom_native""","""c""","""FTMT24400002981.1""","""protein coding mRNAs""","[{""ENSG00000173465.3"",""promoter"",2207.5,2207.5,null,null,1.6900e-14}]"
"""FCHS_73778""","""cp3@ZNRD2-DT""","""PLA""","""chr11""",65569599,65570010,0.895414,449,"[{""EH38E2964563"",""pELS""}]",null,"""ENST00000913761.1""",-49,"""+""","{""ENSG00000173465.9"",""ZNRD2"",""protein_coding"",null,[""basic"", ""GENCODE_Primary"", ""TAGENE""]}",0,"""ENST00000726069.1""",-52,"""+""","{""ENSG00000294808.1"",null,""lncRNA"",""URS0002A38452"",[""basic"", ""Ensembl_canonical"", … ""TAGENE""]}","""same strand""",-101,"""5_c_fantom_native""","""c""","""FTMT24400002981.1""","""protein coding mRNAs""","[{""ENSG00000173465.3"",""promoter"",2207.5,2207.5,null,null,1.6900e-14}]"
"""FCHS_73780""","""cp1@ZNRD2,cp2@ZNRD2-DT""","""PLA""","""chr11""",65570054,65570667,0.985004,0,"[{""EH38E2964564"",""PLS""}, {""EH38E2964565"",""PLS""}, {""EH38E2964566"",""pELS""}]",null,"""ENST00000913761.1""",-49,"""+""","{""ENSG00000173465.9"",""ZNRD2"",""protein_coding"",null,[""basic"", ""GENCODE_Primary"", ""TAGENE""]}",-224,"""ENST00000726069.1""",-52,"""+""","{""ENSG00000294808.1"",null,""lncRNA"",""URS0002A38452"",[""basic"", ""Ensembl_canonical"", … ""TAGENE""]}","""same strand""",-101,"""5_c_fantom_native""","""c""","""FTMT24400002981.1""","""protein coding mRNAs""","[{""ENSG00000173465.3"",""promoter"",2207.5,2207.5,null,null,1.6900e-14}]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_509213""","""cp5@ZNF271P""","""ELA""","""chr18""",35289825,35289927,0.875,-270,"[{""EH38E3263492"",""pELS""}]",null,"""ENST00000589178.5""",-48,"""-""","{""ENSG00000186814.16"",""ZSCAN30"",""protein_coding"",null,[""basic""]}",-274,"""ENST00000586922.2""",-172,"""-""","{""ENSG00000268573.1"",null,""lncRNA"",""URS00002F4E7E"",[""basic"", ""Ensembl_canonical"", ""GENCODE_Primary""]}

&nbsp;

## 2-7-2: Divergent Transcript (stringent selection criteria)

- 以下の条件を満たすペアを選抜

&nbsp;

1. protein-coding genes and lncRNA transcripts are located on the oppsite strand
2. `Promoter Level Activity (PLA)` entry

- Flag (not a filter): whether SCREEN also supports promoter classification (`PLS`) for a cCRE overlapping the fanta.bio CRE. This is stored in `screen_promoter_support`.
- Flag (not a filter): whether a GENCODE/Ensembl promoter window maps to this fanta.bio CRE in `ensembl_promoter_intersect`. This is stored in `ensembl_promoter_support`.
  - related notebook: [../02_fantabio/02_4_ensembl_promoter_intersect.ipynb](../02_fantabio/02_4_ensembl_promoter_intersect.ipynb)


In [24]:
common_cre_divergent = common_cre_fantomcat_mapping.filter(
    (pl.col("strand_type") == "opposite strand") & # condition 1
    (pl.col("fantabio_cre_class") == "PLA") # condition 2
).with_columns(

    # flag1: SCREEN also supports promoter (PLS)
    pl.when(
        pl.col("screen_ccres")
        .list.eval(pl.element().struct.field("screen_ccre_type") == "PLS")
        .list.any()
        .fill_null(False)
    ).then(
        pl.lit("supported by SCREEN")
    ).otherwise(
        pl.lit("not supported by SCREEN")
    ).alias("screen_promoter_support"),


    # flag2: CRE is the representative fanta.bio overlap of a GENCODE/Ensembl promoter window
    pl.when(
        pl.col("cre_id").is_in(pl.lit(ensembl_promoter_intersect.get_column("cre_id")).implode())
        .fill_null(False)
    ).then(
        pl.lit("supported by Ensembl/GENCODE promoter window")
    ).otherwise(
        pl.lit("not supported by Ensembl/GENCODE promoter window")
    ).alias("ensembl_promoter_window_support"),


    # flag3: FANTOM-CAT support
    pl.when(
        pl.col("gene class") == "lncRNA, divergent"
    ).then(
        pl.lit("supported by FANTOM-CAT (stringent)")
    ).otherwise(
        pl.lit("not supported by FANTOM-CAT (stringent)")
    ).alias("fantomcat_stringent_support")
).sort(
    by="pc_lncrna_HN2",
    descending=True
)

common_cre_divergent.write_parquet(
    "../../out/transcript_classification/divergent_transcript/pc_common_cre_lncRNA_divergent/down5p_fantabio_pc_common_cre_lncRNA_divergent.parquet"
)

pl.Config.set_tbl_rows(10)
print(f"unique divergent CRE count: {common_cre_divergent.select(['cre_id']).n_unique()}")
print(f"unique protein-coding transcript count: {common_cre_divergent.select(['pc_transcript_id']).n_unique()}")
print(f"unique lncRNA transcript count: {common_cre_divergent.select(['lncrna_transcript_id']).n_unique()}")
print(f"unique lncRNA - CRE pair count: {common_cre_divergent.select(['cre_id', 'lncrna_transcript_id']).n_unique()}")
display(common_cre_divergent)


unique divergent CRE count: 910
unique protein-coding transcript count: 963
unique lncRNA transcript count: 743
unique lncRNA - CRE pair count: 1344


cre_id,cre_name,fantabio_cre_class,cre_chrom,cre_chrom_start,cre_chrom_end,directionality,pc_transcript_distance,screen_ccres,fantom5_enhancers,pc_transcript_id,pc_HN2,pc_transcript_strand,pc_transcript_info,lncrna_transcript_distance,lncrna_transcript_id,lncrna_HN2,lncrna_transcript_strand,lncrna_transcript_info,strand_type,pc_lncrna_HN2,id_correspondence_flag,Transcript_classification_codes,FANTOM-CAT_transcript_lv4_stringent,gene class,fantomcat_transcript_info,screen_promoter_support,ensembl_promoter_window_support,fantomcat_stringent_support
str,str,str,str,i64,i64,f64,i64,list[struct[2]],list[struct[1]],str,i64,str,struct[5],i64,str,i64,str,struct[5],str,i64,str,str,str,str,list[struct[7]],str,str,str
"""FCHS_267571""","""cp1@GATD3""","""PLA""","""chr21""",44133549,44133713,1.0,0,"[{""EH38E3462368"",""PLS""}]",null,"""ENST00000348499.9""",-48,"""+""","{""ENSG00000160221.19"",""GATD3"",""protein_coding"",null,[""basic"", ""GENCODE_Primary"", ""CCDS""]}",66,"""ENST00000757773.1""",-49,"""-""","{""ENSG00000298760.1"",null,""lncRNA"",""URS0002A53420"",[""basic"", ""Ensembl_canonical"", … ""TAGENE""]}","""opposite strand""",-97,null,null,null,null,null,"""supported by SCREEN""","""supported by Ensembl/GENCODE promoter window""","""not supported by FANTOM-CAT (stringent)"""
"""FCHS_203786""","""cp1@ATP13A1""","""PLA""","""chr19""",19663487,19664011,-0.950446,0,"[{""EH38E3297556"",""pELS""}, {""EH38E3297557"",""PLS""}, {""EH38E3297558"",""pELS""}]",null,"""ENST00000888077.1""",-48,"""-""","{""ENSG00000105726.18"",""ATP13A1"",""protein_coding"",null,[""basic"", ""TAGENE""]}",0,"""ENST00000730427.1""",-49,"""+""","{""ENSG00000295487.1"",null,""lncRNA"",""URS0002A6AA5D"",[""basic"", ""Ensembl_canonical"", … ""TAGENE""]}","""opposite strand""",-97,null,null,null,null,null,"""supported by SCREEN""","""not supported by Ensembl/GENCODE promoter window""","""not supported by FANTOM-CAT (stringent)"""
"""FCHS_203785""","""cp4@ATP13A1""","""PLA""","""chr19""",19662943,19663443,0.861316,-256,"[{""EH38E1945465"",""pELS""}, {""EH38E3297555"",""pELS""}, {""EH38E3297556"",""pELS""}]",null,"""ENST00000888077.1""",-48,"""-""","{""ENSG00000105726.18"",""ATP13A1"",""protein_coding"",null,[""basic"", ""TAGENE""]}",272,"""ENST00000730427.1""",-49,"""+""","{""ENSG00000295487.1"",null,""lncRNA"",""URS0002A6AA5D"",[""basic"", ""Ensembl_canonical"", … ""TAGENE""]}","""opposite strand""",-97,null,null,null,null,null,"""not supported by SCREEN""","""not supported by Ensembl/GENCODE promoter window""","""not supported by FANTOM-CAT (stringent)"""
"""FCHS_416384""","""cp2@INCR1""","""PLA""","""chr9""",5628416,5628978,0.932283,47,"[{""EH38E2679873"",""pELS""}, {""EH38E3873030"",""PLS""}, {""EH38E4489849"",""pELS""}]",null,"""ENST00000251879.10""",-48,"""+""","{""ENSG00000107036.13"",""RIC1"",""protein_coding"",null,[""basic"", ""CCDS""]}",-27,"""ENST00000687545.3""",-49,"""-""","{""ENSG00000286162.5"",null,""lncRNA"",""URS0002A602F7"",[""TAGENE""]}","""opposite strand""",-97,null,null,null,null,null,"""supported by SCREEN""","""supported by Ensembl/GENCODE promoter window""","""not supported by FANTOM-CAT (stringent)"""
"""FCHS_416385""","""cp1@RIC1""","""PLA""","""chr9""",5628738,5629307,0.915455,0,"[{""EH38E3873030"",""PLS""}, {""EH38E3873031"",""pELS""}, {""EH38E4489849"",""pELS""}]",null,"""ENST00000251879.10""",-48,"""+""","{""ENSG00000107036.13"",""RIC1"",""protein_coding"",null,[""basic"", ""CCDS""]}",0,"""ENST00000687545.3""",-49,"""-""","{""ENSG00000286162.5"",null,""lncRNA"",""URS0002A602F7"",[""TAGENE""]}","""opposite strand""",-97,null,null,null,null,null,"""supported by SCREEN""","""not supported by Ensembl/GENCODE promoter window""","""not supported by FANTOM-CAT (stringent)"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_403343""","""cp1@PCMTD1,cp3@PCMTD1-DT""","""PLA""","""chr8""",51898933,51899557,-0.882828,0,"[{""EH38E2630615"",""PLS""}, {""EH38E2630616"",""PLS""}, {""EH38E2630617"",""PLS""}]",null,"""ENST00000521344.1

&nbsp;

## 2-7-3: Divergent Transcript (but Enhancer level activity)

- 既に`common_cre_divergent`でペアが定義されているものも含まれていることに注意
- (e.g., FCHS_361737 の他に、FCHS_361738 のペアも近傍にあるが、FCHS_361738 のペアは ELA のため、`common_cre_divergent`には含まれていない)


In [25]:
common_cre_divergent_ela = common_cre_fantomcat_mapping.filter(
    (pl.col("strand_type") == "opposite strand") &
    (pl.col("fantabio_cre_class") == "ELA")
).sort(
    by="cre_id",
    descending=True
).with_columns(
    pl.when(
        # flag: SCREEN also supports promoter (PLS)
        pl.col("screen_ccres")
        .list.eval(pl.element().struct.field("screen_ccre_type") == "pELS")
        .list.any()
        .fill_null(False)
    ).then(
        pl.lit("supported by SCREEN")
    ).otherwise(
        pl.lit("not supported by SCREEN")
    ).alias("screen_pELS_support"),


    # flag: CRE is the representative fanta.bio overlap of a GENCODE/Ensembl promoter window
    pl.when(
        pl.col("cre_id").is_in(pl.lit(ensembl_promoter_intersect.get_column("cre_id")).implode())
        .fill_null(False)
    ).then(
        pl.lit("supported by Ensembl/GENCODE promoter window")
    ).otherwise(
        pl.lit("not supported by Ensembl/GENCODE promoter window")
    ).alias("ensembl_promoter_window_support"),


    # flag: FANTOM-CAT support
    pl.when(
        pl.col("gene class") == "lncRNA, divergent"
    ).then(
        pl.lit("supported by FANTOM-CAT (stringent)")
    ).otherwise(
        pl.lit("not supported by FANTOM-CAT (stringent)")
    ).alias("fantomcat_stringent_support")
)

pl.Config.set_tbl_rows(10)
print(f"unique divergent CRE count: {common_cre_divergent_ela.select(['cre_id']).n_unique()}")
print(f"unique protein-coding transcript count: {common_cre_divergent_ela.select(['pc_transcript_id']).n_unique()}")
print(f"unique lncRNA transcript count: {common_cre_divergent_ela.select(['lncrna_transcript_id']).n_unique()}")
display(common_cre_divergent_ela)


unique divergent CRE count: 573
unique protein-coding transcript count: 651
unique lncRNA transcript count: 487


cre_id,cre_name,fantabio_cre_class,cre_chrom,cre_chrom_start,cre_chrom_end,directionality,pc_transcript_distance,screen_ccres,fantom5_enhancers,pc_transcript_id,pc_HN2,pc_transcript_strand,pc_transcript_info,lncrna_transcript_distance,lncrna_transcript_id,lncrna_HN2,lncrna_transcript_strand,lncrna_transcript_info,strand_type,pc_lncrna_HN2,id_correspondence_flag,Transcript_classification_codes,FANTOM-CAT_transcript_lv4_stringent,gene class,fantomcat_transcript_info,screen_pELS_support,ensembl_promoter_window_support,fantomcat_stringent_support
str,str,str,str,i64,i64,f64,i64,list[struct[2]],list[struct[1]],str,i64,str,struct[5],i64,str,i64,str,struct[5],str,i64,str,str,str,str,list[struct[7]],str,str,str
"""FCHS_9688""","""cp2@IQCC-AS1""","""ELA""","""chr1""",32206157,32206659,-1.0,-486,"[{""EH38E2800907"",""PLS""}]",null,"""ENST00000291358.11""",-109,"""+""","{""ENSG00000160051.13"",""IQCC"",""protein_coding"",null,[""basic"", ""Ensembl_canonical"", … ""CCDS""]}",0,"""ENST00000373604.4""",-50,"""-""","{""ENSG00000224066.1"",""IQCC-AS1"",""lncRNA"",""URS0000051CC8"",[""basic"", ""Ensembl_canonical"", … ""exp_conf""]}","""opposite strand""",-159,null,null,null,null,null,"""not supported by SCREEN""","""not supported by Ensembl/GENCODE promoter window""","""not supported by FANTOM-CAT (stringent)"""
"""FCHS_9688""","""cp2@IQCC-AS1""","""ELA""","""chr1""",32206157,32206659,-1.0,-486,"[{""EH38E2800907"",""PLS""}]",null,"""ENST00000291358.11""",-109,"""+""","{""ENSG00000160051.13"",""IQCC"",""protein_coding"",null,[""basic"", ""Ensembl_canonical"", … ""CCDS""]}",-155,"""ENST00000421616.1""",-51,"""-""","{""ENSG00000224066.1"",""IQCC-AS1"",""lncRNA"",""URS000044D783"",[""basic""]}","""opposite strand""",-160,null,null,null,null,null,"""not supported by SCREEN""","""not supported by Ensembl/GENCODE promoter window""","""not supported by FANTOM-CAT (stringent)"""
"""FCHS_96241""","""cp8@ANKRD52""","""ELA""","""chr12""",56257898,56258187,-0.986135,-197,"[{""EH38E3020429"",""pELS""}, {""EH38E3020430"",""pELS""}]",null,"""ENST00000267116.8""",-57,"""-""","{""ENSG00000139645.11"",""ANKRD52"",""protein_coding"",null,[""basic"", ""Ensembl_canonical"", … ""CCDS""]}",-293,"""ENST00000770756.1""",-49,"""+""","{""ENSG00000300314.1"",null,""lncRNA"",""URS0002A35471"",[""basic"", ""TAGENE""]}","""opposite strand""",-106,null,null,null,null,null,"""supported by SCREEN""","""not supported by Ensembl/GENCODE promoter window""","""not supported by FANTOM-CAT (stringent)"""
"""FCHS_96240""","""cp2@ANKRD52""","""ELA""","""chr12""",56257884,56258078,-1.0,-306,"[{""EH38E3020429"",""pELS""}, {""EH38E3020430"",""pELS""}]",null,"""ENST00000267116.8""",-57,"""-""","{""ENSG00000139645.11"",""ANKRD52"",""protein_coding"",null,[""basic"", ""Ensembl_canonical"", … ""CCDS""]}",-279,"""ENST00000770756.1""",-49,"""+""","{""ENSG00000300314.1"",null,""lncRNA"",""URS0002A35471"",[""basic"", ""TAGENE""]}","""opposite strand""",-106,null,null,null,null,null,"""supported by SCREEN""","""not supported by Ensembl/GENCODE promoter window""","""not supported by FANTOM-CAT (stringent)"""
"""FCHS_96239""","""cp7@ANKRD52""","""ELA""","""chr12""",56257710,56257926,0.121951,-458,"[{""EH38E3020428"",""pELS""}, {""EH38E3020429"",""pELS""}]",null,"""ENST00000267116.8""",-57,"""-""","{""ENSG00000139645.11"",""ANKRD52"",""protein_coding"",null,[""basic"", ""Ensembl_canonical"", … ""CCDS""]}",-105,"""ENST00000770756.1""",-49,"""+""","{""ENSG00000300314.1"",null,""lncRNA"",""URS0002A35471"",[""basic"", ""TAGENE""]}","""opposite strand""",-106,null,null,null,null,null,"""supported by SCREEN""","""not supported by Ensembl/GENCODE promoter window""","""not supported by FANTOM-CAT (stringent)"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_101875""","""cp3@TMPO""","""ELA""","""chr12""",98515773,98516157,0.800384,-155,"[{""EH38E1636590"",""PLS""}, {""EH38E4072888"",""PLS""}, {""EH38E4072889"",""PLS""}]",null,"""ENST00000343315.9""",-74,"""+""","{""ENSG00000120802.16"",""T